# Magnetization dynamics and the Landau–Lifshitz–Gilbert equation

**Professor Nano Labs · Spintronics**

**Try it live in your browser:** [Magnetization dynamics lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/magnetization-dynamics/)
**Watch first:** [Magnetization dynamics and the Landau-Lifshitz-Gilbert equation](https://www.youtube.com/watch?v=8fkmn7GJMoc)
**Watch later:** [Spintronics memories devices, how are information stored?](https://www.youtube.com/watch?v=ugrsqPkQXNY) · [Magnetization switching through spin transfer torque](https://www.youtube.com/watch?v=R9z5s225i1U) · [Theory of the tunneling magnetoresistance](https://www.youtube.com/watch?v=wEox88pAC7Q)

A small magnet with all its spins locked together is described by one unit vector, $\mathbf m=\mathbf M/M_S$. It obeys the Landau–Lifshitz–Gilbert (LLG) equation,
$$\frac{d\mathbf m}{dt}=-\gamma\,\mathbf m\times\mathbf H_{\rm eff}+\alpha\,\mathbf m\times\frac{d\mathbf m}{dt},\qquad \mathbf H_{\rm eff}=-\frac{1}{\mu_0M_S}\frac{\partial E}{\partial\mathbf m},$$
a field torque that conserves energy and a damping torque that dissipates it. Taking $\mathbf m\times$ of both sides and using $\mathbf m\cdot d\mathbf m/dt=0$ gives the same equation solved for $d\mathbf m/dt$ (the Landau–Lifshitz form), which is what we integrate:
$$\frac{d\mathbf m}{dt}=-\frac{\gamma}{1+\alpha^2}\Big[\mathbf m\times\mathbf H_{\rm eff}+\alpha\,\mathbf m\times(\mathbf m\times\mathbf H_{\rm eff})\Big].$$
Fields are in A/m, so $\gamma=\mu_0\gamma_e=2.21\times10^5$ m A⁻¹ s⁻¹. We take $\mu_0M_S=1.6$ T (CoFeB-like; the video does not give a value), which with the video's $K_{\rm eff}=4\times10^5$ J/m³ gives $H_K=5\times10^5$ A/m. The four sections match the lab's four tabs. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp, quad
from scipy.constants import mu_0, physical_constants

gamma_e = physical_constants['electron gyromag. ratio'][0]   # rad / (s T)
gamma = mu_0 * gamma_e                                     # m / (A s), multiplies H in A/m
Ms = 1.6 / mu_0                                            # A/m  (mu0 Ms = 1.6 T)
HK = lambda K: 2 * K / (mu_0 * Ms)                         # anisotropy field, A/m
deg = np.pi / 180
print(f'gamma = {gamma:.4e} m/(A s) = {gamma / mu_0 / (2 * np.pi) / 1e9:.2f} GHz per tesla')
print(f'Ms = {Ms:.4e} A/m;  H_K for K = 4e5 J/m^3: {HK(4e5):.4e} A/m (mu0 H_K = {mu_0 * HK(4e5):.3f} T)')

def llg(t, m, field, K, alpha):
    # dm/dt in the Landau-Lifshitz form, for an applied field(t) (A/m) plus uniaxial anisotropy along z
    H = np.asarray(field(t), float) + np.array([0, 0, HK(K) * m[2]])
    mxH = np.cross(m, H)
    return -gamma / (1 + alpha ** 2) * (mxH + alpha * np.cross(m, mxH))

def run(m0, field, K, alpha, T, n=3000, **kw):
    m0 = np.asarray(m0, float) / np.linalg.norm(m0)
    s = solve_ivp(llg, (0, T), m0, args=(field, K, alpha), t_eval=np.linspace(0, T, n),
                  rtol=1e-9, atol=1e-11, method='DOP853', **kw)
    return s.t, s.y

angles = lambda th, ph=0: np.array([np.sin(th) * np.cos(ph), np.sin(th) * np.sin(ph), np.cos(th)])

## 1 · Precession

With no damping and a field $H$ along $z$, $d\mathbf m/dt=-\gamma\,\mathbf m\times\mathbf H$ is perpendicular to both $\mathbf m$ and $\mathbf H$. The angle $\theta$ to the field, and with it the energy $E=-\mu_0M_SH\cos\theta$, never change: $\mathbf m$ circles the field at $f=\gamma H/2\pi$, whatever the angle. We check this for the video's two fields, $10^5$ and $5\times10^5$ A/m, and two starting angles.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for H, th0, ls in [(1e5, 30, '-'), (5e5, 30, '--'), (1e5, 150, ':')]:
    t, m = run(angles(th0 * deg), lambda t: [0, 0, H], 0, 0, 1e-9)
    up = np.where((m[0][:-1] < 0) & (m[0][1:] >= 0))[0]           # m_x crossing zero upward, once a turn
    tc = t[up] - m[0][up] * (t[up + 1] - t[up]) / (m[0][up + 1] - m[0][up])
    f_sim, f_th = 1 / np.diff(tc).mean(), gamma * H / (2 * np.pi)
    E = -np.cos(np.arccos(m[2]))
    print(f'H = {H:.0e} A/m, theta0 = {th0:3d} deg: f = {f_sim / 1e9:.4f} GHz (gamma H/2pi = {f_th / 1e9:.4f}),'
          f' period {1e12 / f_th:.1f} ps;  E/mu0MsH stays at {E.mean():+.4f} (spread {np.ptp(E):.1e})')
    ax[0].plot(t * 1e9, m[0], ls, label=f'm_x, H = {H:.0e}, θ0 = {th0}°')
ax[0].set(xlabel='time (ns)', ylabel='$m_x$'); ax[0].legend(fontsize=8)
Hs = np.linspace(0, 1e6, 50)
ax[1].plot(Hs / 1e5, gamma * Hs / (2 * np.pi) / 1e9)
ax[1].set(xlabel='H ($10^5$ A/m)', ylabel='f (GHz)', title='28 GHz per tesla of $\\mu_0H$')
plt.tight_layout(); plt.show()

The same motion on the sphere, as on the video's slides:

In [ ]:
def sphere_axes(ax):
    u, v = np.mgrid[0:2 * np.pi:40j, 0:np.pi:20j]
    ax.plot_wireframe(np.cos(u) * np.sin(v), np.sin(u) * np.sin(v), np.cos(v), color='0.85', lw=0.4)
    ax.plot([0, 0], [0, 0], [-1.2, 1.2], 'g--', lw=1)
    ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()

fig = plt.figure(figsize=(10, 3.6))
for i, (H, th0, alpha, title) in enumerate([(1e5, 30, 0, 'α = 0: precession'), (5e5, 30, 0.1, 'α = 0.1: a spiral in'), (5e5, 150, 0.1, 'α = 0.1 from 150°')]):
    t, m = run(angles(th0 * deg), lambda t: [0, 0, H], 0, alpha, 1e-9 if alpha == 0 else 0.5e-9)
    ax = fig.add_subplot(1, 3, i + 1, projection='3d'); sphere_axes(ax)
    ax.plot(*m, color='tab:red', lw=1.2); ax.quiver(0, 0, 0, *m[:, -1], color='k', lw=2)
    ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

## 2 · Damping

With damping, a second part of the motion points toward the field. For a field along $z$ the LL form gives $d\theta/dt=-\alpha\gamma H\sin\theta/(1+\alpha^2)$, so
$$\tan\frac{\theta}{2}=\tan\frac{\theta_0}{2}\,e^{-t/\tau},\qquad m_z=\tanh\!\Big(\frac t\tau+\tanh^{-1}\cos\theta_0\Big),\qquad \tau=\frac{1+\alpha^2}{\alpha\gamma H},$$
while the precession slows to $\gamma H/2\pi(1+\alpha^2)$. Each $\tau$ lasts $1/2\pi\alpha$ turns, and $\tau$ is shortest at $\alpha=1$.

In [ ]:
H, th0 = 5e5, 30 * deg
tau = lambda a: (1 + a * a) / (a * gamma * H)
mz_exact = lambda t, a: np.tanh(t / tau(a) + np.arctanh(np.cos(th0)))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a in (0.1, 1, 10):
    t, m = run(angles(th0), lambda t: [0, 0, H], 0, a, 0.4e-9)
    err = np.abs(m[2] - mz_exact(t, a)).max()
    t1 = tau(a) * np.log(np.tan(th0 / 2) / np.tan(0.5 * deg))
    print(f'alpha = {a:4}: tau = {tau(a) * 1e12:6.1f} ps, within 1 deg after {t1 * 1e12:6.1f} ps'
          f' = {gamma * H / (2 * np.pi * (1 + a * a)) * t1:5.2f} turns;  max |m_z - formula| = {err:.1e}')
    ax[0].plot(t * 1e12, m[2], label=f'α = {a}')
ax[0].set(xlabel='time (ps)', ylabel='$m_z$', title='$H=5\\times10^5$ A/m, $\\theta_0=30°$'); ax[0].legend()
al = np.logspace(-2, 1, 200)
ax[1].loglog(al, tau(al) * 1e12); ax[1].axvline(1, ls='--', c='g')
ax[1].set(xlabel='α', ylabel='τ (ps)', title='fastest at α = 1; α and 1/α are equally slow')
plt.tight_layout(); plt.show()

## 3 · Switching a bit

Uniaxial anisotropy along $z$ gives two wells, up and down. With a field $H$ up the easy axis,
$$\frac{E}{K_{\rm eff}}=\sin^2\theta-2h\cos\theta,\qquad h=\frac{H}{H_K},\qquad H_K=\frac{2K_{\rm eff}}{\mu_0M_S}.$$
The down state ($\theta=\pi$) is a well while $h\lt1$, behind a barrier $\Delta E=K_{\rm eff}(1-h)^2$ at $\cos\theta_b=-h$. Below we draw the landscape for the video's fields and simulate a bit that starts 5° off straight down, with $\alpha=0.1$.

In [ ]:
K, alpha = 4e5, 0.1
th = np.linspace(-0.15, 1.15, 400) * np.pi
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for H in (0, 1e5, 6e5):
    h = H / HK(K)
    ax[0].plot(th / np.pi, np.sin(th) ** 2 - 2 * h * np.cos(th), label=f'H = {H:.0e} A/m (h = {h:.1f})')
    barrier = K * (1 - h) ** 2 if h < 1 else 0
    print(f'H = {H:.0e} A/m: h = {h:.2f}, barrier {barrier:.3e} J/m^3 = {barrier / K:.0%} of K')
ax[0].set(xlabel='θ/π', ylabel='E/$K_{eff}$'); ax[0].legend(fontsize=8)
for H in (1e5, 6e5):
    t, m = run(angles(np.pi - 5 * deg), lambda t: [0, 0, H], K, alpha, 2.5e-9)
    ax[1].plot(t * 1e9, m[2], label=f'H = {H:.0e} A/m')
ax[1].set(xlabel='time (ns)', ylabel='$m_z$'); ax[1].axhline(0, c='0.7', lw=0.8); ax[1].legend()
plt.tight_layout(); plt.show()

Because the motion stays symmetric about $z$, $m_z=u$ obeys a one-dimensional equation that integrates exactly by partial fractions:
$$\frac{du}{dt}=\frac{\alpha\gamma}{1+\alpha^2}\,(H+H_Ku)(1-u^2),\qquad t(u_0\to u_1)=\frac{1+\alpha^2}{\alpha\gamma}\big[F(u_1)-F(u_0)\big],$$
$$F(u)=\frac{H_K}{H_K^2-H^2}\ln|H+H_Ku|-\frac{\ln(1-u)}{2(H+H_K)}+\frac{\ln(1+u)}{2(H-H_K)}.$$
Near the down pole the tilt grows as $e^{t/t_0}$ with $t_0=(1+\alpha^2)/\alpha\gamma(H-H_K)$, so the switching time grows as the logarithm of the starting tilt.

In [ ]:
def F(u, H, K):
    a, b = H, HK(K)
    return b / (b * b - a * a) * np.log(abs(a + b * u)) - np.log(1 - u) / (2 * (a + b)) + np.log(1 + u) / (2 * (a - b))
def t_axis(H, K, alpha, u0, u1):
    return (1 + alpha ** 2) / (alpha * gamma) * (F(u1, H, K) - F(u0, H, K))

for tilt in (5, 2.5, 1):
    u0 = -np.cos(tilt * deg)
    tc = t_axis(6e5, K, alpha, u0, 0)
    t, m = run(angles(np.pi - tilt * deg), lambda t: [0, 0, 6e5], K, alpha, 2 * tc, n=20001)
    i = np.argmax(m[2] > 0); t_sim = t[i - 1] - m[2][i - 1] * (t[i] - t[i - 1]) / (m[2][i] - m[2][i - 1])
    print(f'tilt {tilt:3} deg: crosses the equator at {tc * 1e9:.4f} ns (formula), {t_sim * 1e9:.4f} ns (simulation)')
t0 = (1 + alpha ** 2) / (alpha * gamma * (6e5 - HK(K)))
print(f't0 = {t0 * 1e9:.3f} ns; halving the tilt adds about t0 ln 2 = {t0 * np.log(2) * 1e9:.3f} ns')

Hs = np.linspace(HK(K) * 1.001, 1e6, 300)
plt.figure(figsize=(5.5, 3.4))
for tilt in (5, 1):
    plt.plot(Hs / 1e5, [t_axis(H, K, alpha, -np.cos(tilt * deg), 0) * 1e9 for H in Hs], label=f'{tilt}° tilt')
plt.axvline(HK(K) / 1e5, ls='--', c='tab:orange'); plt.ylim(0, 4)
plt.xlabel('H ($10^5$ A/m)'); plt.ylabel('time to the equator (ns)'); plt.legend(); plt.tight_layout(); plt.show()

**A pulse.** With the field off, the landscape is the symmetric double well and its barrier is the equator. So a pulse writes the bit only if it lasts until the magnetization has crossed the equator. The step at $t_p$ is handled by integrating the two halves separately.

In [ ]:
def pulse(tp, H=6e5, tilt=5, T=4e-9):
    t1, m1 = run(angles(np.pi - tilt * deg), lambda t: [0, 0, H], K, alpha, tp, n=800)
    t2, m2 = run(m1[:, -1], lambda t: [0, 0, 0], K, alpha, T - tp, n=800)
    return np.concatenate([t1, tp + t2[1:]]), np.concatenate([m1, m2[:, 1:]], axis=1), m1[2, -1]
plt.figure(figsize=(5.5, 3.4))
for tp in (0.8e-9, 1.0e-9):
    t, m, mz_p = pulse(tp)
    print(f'pulse {tp * 1e9:.1f} ns: m_z = {mz_p:+.3f} when the field stops, final m_z = {m[2, -1]:+.3f}')
    plt.plot(t * 1e9, m[2], label=f'{tp * 1e9:.1f} ns pulse')
plt.axhline(0, c='0.7', lw=0.8); plt.xlabel('time (ns)'); plt.ylabel('$m_z$'); plt.legend(); plt.tight_layout(); plt.show()

## 4 · Writing a bit in an MRAM array

In a cross-point array the selected bit feels the word line's field along the easy axis, $H_e$, and the bit line's along the hard axis, $H_x$; its neighbours feel one or the other. For a uniaxial macrospin the down state survives as long as the field lies inside the Stoner–Wohlfarth astroid,
$$\Big|\frac{H_x}{H_K}\Big|^{2/3}+\Big|\frac{H_e}{H_K}\Big|^{2/3}\lt1 .$$
We check it by simulation, with the lab's pulse: a 0.5 ns linear ramp, 2 ns at full field, a 0.5 ns ramp down and 1 ns to settle, starting 1° off straight down. (This takes a few tens of seconds.)

In [ ]:
tr, th_, T = 0.5e-9, 2e-9, 4e-9
env = lambda t: t / tr if t < tr else 1 if t < tr + th_ else 1 - (t - tr - th_) / tr if t < 2 * tr + th_ else 0
def write(hx, he, alpha=0.1, K=4e5):
    f = lambda t: [hx * HK(K) * env(t), 0, he * HK(K) * env(t)]
    s = solve_ivp(llg, (0, T), angles(np.pi - 1 * deg, 30 * deg), args=(f, K, alpha), rtol=1e-8, atol=1e-10, method='DOP853', max_step=5e-12)
    return s.y[2, -1]

grid = np.linspace(0, 1.2, 9)
out = np.array([[write(hx, he) for hx in grid] for he in grid])
p = np.linspace(0, np.pi / 2, 200)
plt.figure(figsize=(4.8, 4.4))
X, Y = np.meshgrid(grid, grid)
plt.scatter(X.ravel(), Y.ravel(), c=np.where(out > 0, 'tab:blue', '0.6').ravel(), s=40)
plt.plot(np.cos(p) ** 3, np.sin(p) ** 3, 'k')
plt.xlabel('$H_x/H_K$ (hard axis)'); plt.ylabel('$H_e/H_K$ (easy axis)'); plt.title('blue: switched')
plt.tight_layout(); plt.show()
for hx, he in [(0.6, 0.6), (0, 0.6), (0.6, 0), (0.4, 0.4), (0.3, 0.3)]:
    print(f'H_x = {hx}, H_e = {he}: astroid sum {hx ** (2 / 3) + he ** (2 / 3):.3f}, final m_z = {write(hx, he):+.3f}')
print(f'equal fields switch above (1/2)^(3/2) = {0.5 ** 1.5:.4f} H_K; for H = 0.6 H_K a 100 nm line needs I = 2wH = {2 * 100e-9 * 0.6 * HK(4e5) * 1e3:.0f} mA')

Near the astroid the simulation and the static boundary differ slightly: just outside it the old state is barely unstable and takes longer than the pulse to escape (most of all on the easy axis, where the torque on a bit pointing straight down starts at zero), and just inside it the ramp can ring the magnetization over the barrier. At $\alpha=0.1$ equal fields switch from about $0.350\,H_K$, against $0.354\,H_K$ for the astroid.

## Checks: the numbers quoted in the lab's answers

In [ ]:
f = lambda H: gamma * H / (2 * np.pi)
# Q1: 3.52 GHz and 284 ps at 1e5 A/m; 17.6 GHz and 57 ps at 5e5; 28.0 GHz per tesla
assert abs(f(1e5) / 1e9 - 3.52) < 0.005 and abs(1e12 / f(1e5) - 284) < 0.5
assert abs(f(5e5) / 1e9 - 17.6) < 0.05 and abs(1e12 / f(5e5) - 57) < 0.5 and abs(gamma_e / (2 * np.pi) / 1e9 - 28.0) < 0.05
# Q2: from 150 deg, no damping: m_z stays at cos 150 = -0.866, E = +0.87 mu0 Ms H
t, m = run(angles(150 * deg), lambda t: [0, 0, 1e5], 0, 0, 1e-9)
assert np.ptp(m[2]) < 1e-6 and abs(m[2].mean() + 0.866) < 0.001
# Q3: alpha = 0.1, 5e5 A/m, 30 deg -> 1 deg: tau = 91 ps, 5.45 turns in 0.31 ns, 1.6 turns per tau
H, a = 5e5, 0.1
tau3 = (1 + a * a) / (a * gamma * H); t1 = tau3 * np.log(np.tan(15 * deg) / np.tan(0.5 * deg))
assert abs(tau3 * 1e12 - 91.3) < 0.1 and abs(t1 * 1e9 - 0.313) < 0.001 and abs(f(H) / (1 + a * a) * t1 - 5.45) < 0.01 and abs(1 / (2 * np.pi * a) - 1.59) < 0.01
# Q4: alpha = 1 takes 62 ps, alpha = 0.1 and 10 both 313 ps
tt = lambda a: (1 + a * a) / (a * gamma * H) * np.log(np.tan(15 * deg) / np.tan(0.5 * deg)) * 1e12
assert abs(tt(1) - 61.9) < 0.1 and abs(tt(0.1) - 312.6) < 0.1 and abs(tt(10) - tt(0.1)) < 1e-9
# Q5: H_K = 5e5 A/m; at 1e5 A/m h = 0.2 and the well is 64% as deep; the bit stays down. At 6e5 it switches
assert abs(HK(4e5) - 5e5) < 1e-3 and abs((1 - 1e5 / HK(4e5)) ** 2 - 0.64) < 1e-9
t, m = run(angles(np.pi - 5 * deg), lambda t: [0, 0, 1e5], 4e5, 0.1, 3e-9)
assert m[2].max() < 0 and m[2, -1] < -0.999
t, m = run(angles(np.pi - 5 * deg), lambda t: [0, 0, 6e5], 4e5, 0.1, 3e-9)
assert m[2, -1] > 0.999
# Q6: 0.92 ns from 5 deg, 1.23 ns from 2.5 deg (+0.31 ns); t0 = 0.46 ns
tc5, tc25 = t_axis(6e5, 4e5, 0.1, -np.cos(5 * deg), 0), t_axis(6e5, 4e5, 0.1, -np.cos(2.5 * deg), 0)
assert abs(tc5 * 1e9 - 0.918) < 0.001 and abs(tc25 * 1e9 - 1.231) < 0.001 and abs((tc25 - tc5) * 1e9 - 0.31) < 0.01
assert abs((1 + 0.01) / (0.1 * gamma * 1e5) * 1e9 - 0.456) < 0.001
# Q7: a 0.8 ns pulse leaves m_z = -0.74 and falls back; a 1.0 ns pulse leaves +0.92 and stays up
_, m8, p8 = pulse(0.8e-9); _, m10, p10 = pulse(1.0e-9)
assert abs(p8 + 0.735) < 0.005 and m8[2, -1] < -0.999 and abs(p10 - 0.916) < 0.005 and m10[2, -1] > 0.999
# Q8: 0.6 + 0.6 is outside the astroid (1.42), each alone inside; equal fields need 0.354; 60 mA in a 100 nm line
assert abs(2 * 0.6 ** (2 / 3) - 1.42) < 0.005 and abs(0.5 ** 1.5 - 0.354) < 0.001
assert write(0.6, 0.6) > 0.99 and write(0, 0.6) < -0.99 and write(0.6, 0) < -0.99
assert abs(2 * 100e-9 * 0.6 * HK(4e5) * 1e3 - 60) < 1e-6
print("all the lab's answers check out")

## Things to try

1. **Ferromagnetic resonance.** Add a small oscillating field along $x$, $h_0\cos(2\pi f t)$, to a static field along $z$ with $\alpha=0.01$. Sweep $f$ and plot the steady-state amplitude of $m_x$: the peak sits at $\gamma H/2\pi$ and its width measures $\alpha$.
2. **Thermal noise.** Real bits start off their axis because of heat. Add a random field with variance $2\alpha k_BT/(\gamma\mu_0M_SV\Delta t)$ (the Brown field) and look at the spread of switching times at $6\times10^5$ A/m for a 40 nm bit.
3. **A tilted field.** Apply the switching field at 45° to the easy axis. The astroid says it needs only half of $H_K$: check it, and compare how fast it switches with the on-axis field.
4. **Spin-transfer torque.** The video's last slide adds a torque $\frac{\gamma}{\mu_0M_Sd}\mathbf N$. Try $\mathbf N\propto\mathbf m\times(\mathbf m\times\hat{\mathbf p})$ with $\hat{\mathbf p}=+\hat{\mathbf z}$, the Slonczewski form, and find the current that switches the bit with no field at all.